# 01. 무신사 상품 및 리뷰 크롤링

본 노트북은 무신사(Musinsa)에서 지정한 브랜드의 상품 정보와 리뷰 데이터를 수집하는 크롤러입니다.

## 수집 대상
- **브랜드**: `jemut`, `filluminate`, `travel`
- **카테고리**: 상의(`001`), 아우터(`002`), 하의(`003`)

## 수집 항목
- **상품 정보**: 상품명, 정가/판매가, 할인율, 리뷰 수, 리뷰 점수, 판매 상태, 누적 판매수, 조회수
- **리뷰 정보**: 리뷰 내용, 평점, 작성자 프로필(키/몸무게/성별), 만족도, 구매 옵션, 사진 유무, 도움돼요 수, 체험단 여부, 작성일

## 출력 파일
브랜드 × 카테고리 조합별로 `data/` 디렉토리에 다음 CSV 파일을 생성합니다.
- `data/{브랜드}_products_{카테고리}.csv` — 상품 정보
- `data/{브랜드}_reviews_{카테고리}.csv` — 리뷰 정보
- `data/{브랜드}_errors_{카테고리}.csv` — 수집 실패 기록

## 노트북 구성
1. **라이브러리 임포트**
2. **단일 브랜드 버전 (참고용, 전체 주석 처리)** — 다중 브랜드 루프 도입 이전의 초기 버전 코드
3. **메인 크롤러** — 다중 브랜드 × 카테고리 순회 수집 로직
4. **CSV 중복 제거 후처리** — 누적 저장된 CSV 파일에서 중복 행 제거

## 동작 방식 요약
- 무신사의 비공식 JSON API(`api.musinsa.com`, `goods.musinsa.com`, `goods-detail.musinsa.com`)에 직접 요청을 보내는 방식입니다.
- 요청 사이 `time.sleep(2)`로 대기하며, 실패 시 지수 백오프(최대 3회)로 재시도합니다.
- 이미 수집된 상품/리뷰 ID를 기억하여 중복 수집을 회피하는 **재시작 가능 구조**입니다.


---

## 1. 라이브러리 임포트

크롤링과 데이터 처리에 사용되는 기본 라이브러리를 불러옵니다.

- `requests` — HTTP 요청
- `pandas` — 데이터프레임 및 CSV 입출력
- `time` — 요청 간 대기 시간 부여
- `os` — 파일 존재 여부 확인 (재시작 로직에 사용)


In [1]:
import os
import time
import requests
import pandas as pd

In [2]:
# # ==========================================
# # 만족도 파싱 함수
# # ==========================================
# def parse_satisfaction(survey):
#     if not survey:
#         return ""
#     questions = survey.get("questions", [])
#     parts = []
#     for q in questions:
#         attribute = q.get("attribute", "")
#         answers = q.get("answers", [])
#         answer_text = ", ".join([a.get("answerShortText", "") for a in answers])
#         parts.append(f"{attribute}: {answer_text}")
#     return " / ".join(parts)

# # ==========================================
# # 리뷰 수집 함수 (새로운 리뷰만 수집)
# # ==========================================
# def get_reviews(goods_no, target_count, collected_ids, max_pages=9999):
#     reviews = []
#     stop_crawling = False # 크롤링 중단 플래그

#     for page in range(max_pages):
#         if len(reviews) >= target_count or stop_crawling:
#             break
        
#         # 리뷰 요청 보내기
#         url = "https://goods.musinsa.com/api2/review/v1/view/list"
#         params = {
#             "page":              page,
#             "pageSize":          10,
#             "goodsNo":           goods_no,
#             "sort":              "new", # 최신순 정렬
#             "selectedSimilarNo": goods_no,
#             "myFilter":          "false",
#             "hasPhoto":          "false",
#             "isExperience":      "false",
#         }
#         try:
#             json_data, status_code = request_json(url, params=params, timeout=10, retry=3)
#             if not json_data:
#                 print(f"  상태코드 {status_code} -> 중단")
#                 break
                
#             data = json_data.get("data", {})
#             review_list = data.get("list", [])
            
#             if not review_list:
#                 break
                
#             for review in review_list:
#                 review_no = str(review.get("no", ""))
                
#                 # 핵심 로직: 이미 수집된 리뷰 번호를 만나면 더 이상 과거 리뷰를 탐색할 필요가 없음
#                 if review_no in collected_ids:
#                     stop_crawling = True
#                     break 
                
#                 profile = review.get("userProfileInfo") or {}
#                 images  = review.get("images") or []
#                 reviews.append({
#                     "goodsNo":  str(goods_no),
#                     "리뷰번호": review_no, # 이 값이 중복 체크의 기준이 됩니다
#                     "리뷰타입": review.get("type", ""),
#                     "작성자":   profile.get("userNickName", ""),
#                     "리뷰내용": review.get("content", ""),
#                     "평점":     int(review.get("grade", 0)),
#                     "체험단":   review.get("type") == "experience",
#                     "구매옵션": review.get("goodsOption", ""),
#                     "키":       profile.get("userHeight", ""),
#                     "몸무게":   profile.get("userWeight", ""),
#                     "성별":     profile.get("reviewSex", ""),
#                     "작성일":   review.get("createDate", ""),
#                     "만족도":   parse_satisfaction(review.get("reviewSurveySatisfaction")),
#                     "사진유무": len(images) > 0,
#                     "도움돼요": review.get("likeCount", 0),
#                 })
                
#                 if len(reviews) >= target_count:
#                     break
                    
#             total_pages = data.get("page", {}).get("totalPages", 0)
#             if page >= total_pages - 1:
#                 break
                
#             if len(reviews) % 100 == 0 and len(reviews) > 0:
#                 print(f"  새로운 리뷰 {len(reviews)}개 수집 중...")
#             time.sleep(2)
            
#         except Exception as e:
#             print(f"  리뷰 수집 오류: {e}")
#             time.sleep(2)
#             break
            
#     return reviews

# # ==========================================
# # 상품 통계(조회수, 판매량) 수집 함수
# # ==========================================
# def get_goods_stats(goods_no):
#     url = f"https://goods-detail.musinsa.com/api2/goods/{goods_no}/stat"
#     try:
#         json_data, status_code = request_json(url, timeout=10, retry=3)
#         if json_data:
#             sales_count = json_data.get("data", {}).get("purchaseTotal", 0)
#             views_count = json_data.get("data", {}).get("pageViewTotal", 0)
#             return sales_count, views_count
#         else:
#             print(f"  통계 수집 실패 ({goods_no}): {status_code}")
#     except Exception as e:
#         print(f"  통계 수집 오류 ({goods_no}): {e}")
#     return 0, 0

# # ==========================================
# # escape_excel 함수 추가
# # ==========================================
# def escape_excel(text):
#     if isinstance(text, str) and text.startswith(('-', '=', '+', '@', '#')):
#         return "'" + text
#     return text

# # ==========================================
# # 중간 저장 함수
# # ==========================================
# #def save_product(row_dict):
#     df_row = pd.DataFrame([row_dict])
#     if os.path.exists(PRODUCTS_CSV):
#         df_row.to_csv(PRODUCTS_CSV, mode="a", header=False, index=False, encoding="utf-8-sig")
#     else:
#         df_row.to_csv(PRODUCTS_CSV, mode="w", header=True, index=False, encoding="utf-8-sig")
# def save_reviews(reviews):
#     if not reviews:
#         return
#     df_r = pd.DataFrame(reviews)
    
#     # 줄바꿈 문자 제거 (CSV 깨짐 방지)
#     df_r["리뷰내용"] = df_r["리뷰내용"].astype(str).str.replace(r'[\n\r\t]', ' ', regex=True)

#     # 엑셀 오류 방지 처리
#     for col in ['리뷰내용', '작성자']:
#         if col in df_r.columns:
#             df_r[col] = df_r[col].apply(escape_excel)
    
#     if os.path.exists(REVIEWS_CSV):
#         df_r.to_csv(REVIEWS_CSV, mode="a", header=False, index=False, encoding="utf-8-sig")
#     else:
#         df_r.to_csv(REVIEWS_CSV, mode="w", header=True, index=False, encoding="utf-8-sig")
# def save_error(goods_no, error_type, message):
#     df_e = pd.DataFrame([{
#         "goodsNo":    goods_no,
#         "error_type": error_type,
#         "message":    message,
#         "time":       pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
#     }])
#     if os.path.exists(ERRORS_CSV):
#         df_e.to_csv(ERRORS_CSV, mode="a", header=False, index=False, encoding="utf-8-sig")
#     else:
#         df_e.to_csv(ERRORS_CSV, mode="w", header=True, index=False, encoding="utf-8-sig")
        

# headers = {
#     "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36",
#     "Referer": "https://www.musinsa.com/",
#     "Origin": "https://www.musinsa.com",
# }

# BRAND         = "travel"
# CATEGORIES = {
#     "001": "상의",
#     "002": "아우터",
#     "003": "하의",
# }
# PAGE_SIZE     = 30 # 추측 최대 상한선이 100
# #001 : 상의, 002 : 아우터, 003 : 하의

# session = requests.Session()
# session.headers.update(headers)
# session.headers["Referer"] = f"https://www.musinsa.com/brand/{BRAND}"

# def request_json(url, params=None, timeout=10, retry=3):
#     last_status = None
#     last_error = None
#     for attempt in range(1, retry + 1):
#         try:
#             response = session.get(url, params=params, timeout=timeout)
#             last_status = response.status_code
#             if response.status_code == 200:
#                 return response.json(), response.status_code
#             print(f"요청 실패 ({response.status_code}) -> 재시도 {attempt}/{retry}")
#         except Exception as e:
#             last_error = e
#             print(f"요청 오류 -> 재시도 {attempt}/{retry}: {e}")
#         time.sleep(2 * attempt)
#     if last_error is not None:
#         print(f"최종 요청 오류: {last_error}")
#     return None, last_status

# for CATEGORY, CATEGORY_NAME in CATEGORIES.items():
#     # CSV 경로를 카테고리마다 새로 설정
#     PRODUCTS_CSV = f"{BRAND}_products_{CATEGORY}.csv"
#     REVIEWS_CSV  = f"{BRAND}_reviews_{CATEGORY}.csv"
#     ERRORS_CSV   = f"{BRAND}_errors_{CATEGORY}.csv"

#     print(f"\n{'='*50}")
#     print(f"▶ 카테고리 [{CATEGORY}] {CATEGORY_NAME} 수집 시작")
#     print(f"{'='*50}")
    
#     # ==========================================
#     # 브랜드 전체 상품 수집
#     # ==========================================
#     all_data = []
#     page = 1
#     next_page_url = None  # 다음 페이지 URL (hmacId 포함)

#     while True:
#         try:
#             if next_page_url:
#                 # 2페이지부터: nextPageUrl을 그대로 사용 (hmacId 포함)
#                 json_data, status_code = request_json(next_page_url, timeout=10, retry=3)
#             else:
#                 # 1페이지: 기존 방식으로 요청
#                 url = "https://api.musinsa.com/api2/dp/v2/plp/goods"
#                 params = {
#                     "gf":       "A",
#                     "isSoldOut": "true", # 추가
#                     "sortCode": "REVIEW",
#                     "category": CATEGORY,
#                     "brand":    BRAND,
#                     "size":     PAGE_SIZE,
#                     "caller":   "FLAGSHIP",
#                     "page":     page,
#                 }
#                 json_data, status_code = request_json(url, params=params, timeout=10, retry=3)
            
#             if not json_data: # 데이터가 비어있다면 무한루프 탈출
#                 print(f"상품 수집 실패 (page {page}): {status_code}")
#                 break
            
#             data = json_data.get("data", {})
#             goods_list = data.get("list", [])
#             if not goods_list:# 상품 리스트가 비어있다면 더 이상 수집할 데이터가 없다고 판단하여 반복 종료
#                 break
            
#             for item in goods_list:
#                 is_sold_out = item.get("isSoldOut", False)
#                 all_data.append({
#                     "플랫폼":    "무신사",
#                     "카테고리":  CATEGORY_NAME,
#                     "브랜드":    item.get("brandName", ""),
#                     "goodsNo":  str(item.get("goodsNo", "")),
#                     "상품명":    item.get("goodsName", ""),
#                     "정가":      item.get("normalPrice", 0),
#                     "판매가":    item.get("price", 0),
#                     "할인율": item.get("saleRate", 0),
#                     "리뷰수":    item.get("reviewCount", 0),
#                     "리뷰점수":  item.get("reviewScore", 0),
#                     "판매상태":  "SOLD_OUT" if is_sold_out else "SALE",

#                 })

#             pagination = data.get("pagination", {})
#             total_pages = pagination.get("totalPages", 1)
#             print(f"상품 수집 중... {page}/{total_pages} 페이지 (누적 {len(all_data)}개)")
            
#             # 다음 페이지 URL 추출 (hmacId 포함)
#             next_page_url = pagination.get("nextPageUrl")
            
#             if not pagination.get("hasNext", False) or page >= total_pages:
#                 break
#             page += 1
#             time.sleep(2)

#         except Exception as e:
#             print(f"상품 수집 오류 (page {page}): {e}")
#             time.sleep(2)
#             break

#     # 리뷰 많은 순으로 정렬 (전체)
#     df = pd.DataFrame(all_data)
#     df = df.sort_values("리뷰수", ascending=False).reset_index(drop=True)
#     df["조회수"] = 0
#     df["누적판매수"] = 0
#     print(f"\n상품 총 {len(df)}개 수집 완료 (리뷰 많은 순)")
#     print(df[["카테고리", "브랜드", "상품명", "리뷰수"]].to_string(index=False))

#     # ==========================================
#     # 재시작 로직 - 이미 수집된 goodsNo 및 리뷰번호 확인
#     # ==========================================
#     collected_review_ids = set()

#     if os.path.exists(PRODUCTS_CSV): # products csv 파일 존재시
#         df_existing = pd.read_csv(PRODUCTS_CSV)
#         df_existing["goodsNo"] = df_existing["goodsNo"].astype(str)
#         df_existing.set_index("goodsNo", inplace=True)
#         print(f"\n기존 수집된 상품 {len(df_existing)}개 확인 -> 기존 정보 최신화 및 신규 리뷰 수집 예정")
#     else:
#         df_existing = pd.DataFrame() # 빈 데이터프레임 생성
#         print("\n기존 수집 상품 파일 없음 -> 처음부터 수집 시작")

#     if os.path.exists(REVIEWS_CSV): # reviews csv 파일 존재시
#         df_existing_reviews = pd.read_csv(REVIEWS_CSV)
#         collected_review_ids = set(df_existing_reviews["리뷰번호"].astype(str).tolist())
#         print(f"기존 수집된 개별 리뷰 {len(collected_review_ids)}개 확인 -> 중복 리뷰 스킵 예정")

#     # ==========================================
#     # 전체 상품 리뷰 및 통계 수집
#     # ==========================================
#     total_items = len(df)

#     for idx, row in df.iterrows():
#         goods_no     = row["goodsNo"]
#         target_count = int(row["리뷰수"])
#         current_item = idx + 1
        
#         is_new_product = df_existing.empty or goods_no not in df_existing.index # 상품번호가 기존에 없는 번호라면 True(신규 상품)
#         print(f"\n[{current_item}/{total_items}] 처리 중: {str(row['상품명'])[:25]} (총 리뷰 수: {target_count}개)")

#         # 1. 통계 수집 (기존/신규 상관없이 무조건 수집하여 최신화)
#         try:
#             sales, views = get_goods_stats(goods_no)
#         except Exception as e:
#             print(f"  통계 수집 실패: {e}")
#             save_error(goods_no, "stats", str(e))
#             # API 오류 시 기존 데이터가 있다면 유지, 없다면 0으로 처리
#             if not is_new_product:
#                 sales = df_existing.at[goods_no, "누적판매수"]
#                 views = df_existing.at[goods_no, "조회수"]
#             else:
#                 sales, views = 0, 0
#         time.sleep(2)

#         # # 2. Survey 수집 (신규 상품일 경우에만 진행하도록 권장. 필요시 무조건 수집으로 변경 가능)
#         # # 신규 상품인 경우만 진행하도록 설계되어 있음.
#         # if is_new_product:
#         #     try:
#         #         survey_rows = get_survey(goods_no)
#         #         save_survey(survey_rows)
#         #         print(f"  survey {len(survey_rows)}행 저장 완료 (신규)")
#         #     except Exception as e:
#         #         print(f"  survey 수집 실패: {e}")
#         #         save_error(goods_no, "survey", str(e))
#         #     time.sleep(2)

#         # 3. 데이터프레임(df_existing)에 최신 정보 업데이트 또는 추가
#         updated_info = {
#             "플랫폼":    row["플랫폼"],
#             "카테고리":  row["카테고리"],
#             "브랜드":    row["브랜드"],
#             "상품명":    row["상품명"],
#             "정가":      row["정가"],
#             "판매가":    row["판매가"],
#             "할인율": row["할인율"],
#             "조회수":    views,
#             "누적판매수": sales,
#             "리뷰수":    row["리뷰수"],
#             "리뷰점수":  row["리뷰점수"],
#             "판매상태":  row["판매상태"],
#         }
        
#         # Pandas를 이용해 해당 goods_no 행을 최신 정보로 덮어쓰기 (없으면 새로 생성됨)
#         for key, value in updated_info.items():
#             df_existing.at[goods_no, key] = value
            
#         if is_new_product:
#             print(f"  -> 신규 상품 저장 완료 (누적판매: {sales} | 조회수: {views})")
#         else:
#             print(f"  -> 기존 상품 정보 최신화 완료 (누적판매: {sales} | 조회수: {views})")

#         # 4. 리뷰 수집 로직 (기존 코드와 동일)
#         if not goods_no or target_count == 0:
#             print(f"  [스킵] 리뷰 없음")
#         else:
#             print(f"  새로운 리뷰 수집 확인 중...")
#             try:
#                 reviews = get_reviews(goods_no, target_count=target_count, collected_ids=collected_review_ids)
#                 if len(reviews) > 0:
#                     save_reviews(reviews)
#                     collected_review_ids.update([str(r["리뷰번호"]) for r in reviews])
#                     print(f"  -> 새로운 리뷰 {len(reviews)}개 수집 및 추가 완료!")
#                 else:
#                     print(f"  -> 추가된 새로운 리뷰가 없습니다.")
#             except Exception as e:
#                 print(f"  리뷰 수집 실패: {e}")
#                 save_error(goods_no, "review", str(e))
                
#         time.sleep(2)

#     # ==========================================
#     # 최종 최신화된 상품 데이터를 CSV에 덮어쓰기
#     # ==========================================
#     df_existing.reset_index(inplace=True)

#     if 'index' in df_existing.columns:
#         df_existing.rename(columns={'index': 'goodsNo'}, inplace=True)
#     df_existing.to_csv(PRODUCTS_CSV, index=False, encoding="utf-8-sig")
#     print("\n모든 상품 정보 최신화 및 CSV 저장 완료!")

In [3]:
import requests
import pandas as pd
import time
import os

# ==========================================
# 만족도 파싱 함수
# ==========================================
def parse_satisfaction(survey):
    if not survey:
        return ""
    questions = survey.get("questions", [])
    parts = []
    for q in questions:
        attribute = q.get("attribute", "")
        answers = q.get("answers", [])
        answer_text = ", ".join([a.get("answerShortText", "") for a in answers])
        parts.append(f"{attribute}: {answer_text}")
    return " / ".join(parts)

# ==========================================
# 리뷰 수집 함수 (새로운 리뷰만 수집)
# ==========================================
def get_reviews(goods_no, target_count, collected_ids, max_pages=9999):
    reviews = []
    stop_crawling = False

    for page in range(max_pages):
        if len(reviews) >= target_count or stop_crawling:
            break

        url = "https://goods.musinsa.com/api2/review/v1/view/list"
        params = {
            "page":              page,
            "pageSize":          10,
            "goodsNo":           goods_no,
            "sort":              "new",
            "selectedSimilarNo": goods_no,
            "myFilter":          "false",
            "hasPhoto":          "false",
            "isExperience":      "false",
        }
        try:
            json_data, status_code = request_json(url, params=params, timeout=10, retry=3)
            if not json_data:
                print(f"  상태코드 {status_code} -> 중단")
                break

            data = json_data.get("data", {})
            review_list = data.get("list", [])

            if not review_list:
                break

            for review in review_list:
                review_no = str(review.get("no", ""))

                if review_no in collected_ids:
                    stop_crawling = True
                    break

                profile = review.get("userProfileInfo") or {}
                images  = review.get("images") or []
                reviews.append({
                    "goodsNo":  str(goods_no),
                    "리뷰번호": review_no,
                    "리뷰타입": review.get("type", ""),
                    "작성자":   profile.get("userNickName", ""),
                    "리뷰내용": review.get("content", ""),
                    "평점":     int(review.get("grade", 0)),
                    "체험단":   review.get("type") == "experience",
                    "구매옵션": review.get("goodsOption", ""),
                    "키":       profile.get("userHeight", ""),
                    "몸무게":   profile.get("userWeight", ""),
                    "성별":     profile.get("reviewSex", ""),
                    "작성일":   review.get("createDate", ""),
                    "만족도":   parse_satisfaction(review.get("reviewSurveySatisfaction")),
                    "사진유무": len(images) > 0,
                    "도움돼요": review.get("likeCount", 0),
                })

                if len(reviews) >= target_count:
                    break

            total_pages = data.get("page", {}).get("totalPages", 0)
            if page >= total_pages - 1:
                break

            if len(reviews) % 100 == 0 and len(reviews) > 0:
                print(f"  새로운 리뷰 {len(reviews)}개 수집 중...")
            time.sleep(2)

        except Exception as e:
            print(f"  리뷰 수집 오류: {e}")
            time.sleep(2)
            break

    return reviews

# ==========================================
# 상품 통계 수집 함수
# ==========================================
def get_goods_stats(goods_no):
    url = f"https://goods-detail.musinsa.com/api2/goods/{goods_no}/stat"
    try:
        json_data, status_code = request_json(url, timeout=10, retry=3)
        if json_data:
            sales_count = json_data.get("data", {}).get("purchaseTotal", 0)
            views_count = json_data.get("data", {}).get("pageViewTotal", 0)
            return sales_count, views_count
        else:
            print(f"  통계 수집 실패 ({goods_no}): {status_code}")
    except Exception as e:
        print(f"  통계 수집 오류 ({goods_no}): {e}")
    return 0, 0

# ==========================================
# escape_excel 함수
# ==========================================
def escape_excel(text):
    if isinstance(text, str) and text.startswith(('-', '=', '+', '@', '#')):
        return "'" + text
    return text

# ==========================================
# 중간 저장 함수
# ==========================================
def save_reviews(reviews):
    if not reviews:
        return
    df_r = pd.DataFrame(reviews)
    df_r["리뷰내용"] = df_r["리뷰내용"].astype(str).str.replace(r'[\n\r\t]', ' ', regex=True)
    for col in ['리뷰내용', '작성자']:
        if col in df_r.columns:
            df_r[col] = df_r[col].apply(escape_excel)
    if os.path.exists(REVIEWS_CSV):
        df_r.to_csv(REVIEWS_CSV, mode="a", header=False, index=False, encoding="utf-8-sig")
    else:
        df_r.to_csv(REVIEWS_CSV, mode="w", header=True, index=False, encoding="utf-8-sig")

def save_error(goods_no, error_type, message):
    df_e = pd.DataFrame([{
        "goodsNo":    goods_no,
        "error_type": error_type,
        "message":    message,
        "time":       pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
    }])
    if os.path.exists(ERRORS_CSV):
        df_e.to_csv(ERRORS_CSV, mode="a", header=False, index=False, encoding="utf-8-sig")
    else:
        df_e.to_csv(ERRORS_CSV, mode="w", header=True, index=False, encoding="utf-8-sig")

# ==========================================
# 설정
# ==========================================
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36",
    "Referer": "https://www.musinsa.com/",
    "Origin": "https://www.musinsa.com",
}

BRANDS = ["jemut", "filluminate", "travel"]
CATEGORIES = {
    "001": "상의",
    "002": "아우터",
    "003": "하의",
}
PAGE_SIZE = 30

# ==========================================
# 브랜드 × 카테고리 루프
# ==========================================
for BRAND in BRANDS:
    session = requests.Session()
    session.headers.update(headers)
    session.headers["Referer"] = f"https://www.musinsa.com/brand/{BRAND}"

    def request_json(url, params=None, timeout=10, retry=3):
        last_status = None
        last_error = None
        for attempt in range(1, retry + 1):
            try:
                response = session.get(url, params=params, timeout=timeout)
                last_status = response.status_code
                if response.status_code == 200:
                    return response.json(), response.status_code
                print(f"요청 실패 ({response.status_code}) -> 재시도 {attempt}/{retry}")
            except Exception as e:
                last_error = e
                print(f"요청 오류 -> 재시도 {attempt}/{retry}: {e}")
            time.sleep(2 * attempt)
        if last_error is not None:
            print(f"최종 요청 오류: {last_error}")
        return None, last_status

    print(f"\n{'='*60}")
    print(f"▶ 브랜드 [{BRAND}] 수집 시작")
    print(f"{'='*60}")

    for CATEGORY, CATEGORY_NAME in CATEGORIES.items():
        PRODUCTS_CSV = f"data/{BRAND}_products_{CATEGORY}.csv"
        REVIEWS_CSV  = f"data/{BRAND}_reviews_{CATEGORY}.csv"
        ERRORS_CSV   = f"data/{BRAND}_errors_{CATEGORY}.csv"

        print(f"\n{'='*50}")
        print(f"▶ 카테고리 [{CATEGORY}] {CATEGORY_NAME} 수집 시작")
        print(f"{'='*50}")

        all_data = []
        page = 1
        next_page_url = None

        while True:
            try:
                if next_page_url:
                    json_data, status_code = request_json(next_page_url, timeout=10, retry=3)
                else:
                    url = "https://api.musinsa.com/api2/dp/v2/plp/goods"
                    params = {
                        "gf":        "A",
                        "isSoldOut": "true",
                        "sortCode":  "REVIEW",
                        "category":  CATEGORY,
                        "brand":     BRAND,
                        "size":      PAGE_SIZE,
                        "caller":    "FLAGSHIP",
                        "page":      page,
                    }
                    json_data, status_code = request_json(url, params=params, timeout=10, retry=3)

                if not json_data:
                    print(f"상품 수집 실패 (page {page}): {status_code}")
                    break

                data = json_data.get("data", {})
                goods_list = data.get("list", [])
                if not goods_list:
                    break

                for item in goods_list:
                    is_sold_out = item.get("isSoldOut", False)
                    all_data.append({
                        "플랫폼":   "무신사",
                        "카테고리": CATEGORY_NAME,
                        "브랜드":   item.get("brandName", ""),
                        "goodsNo":  str(item.get("goodsNo", "")),
                        "상품명":   item.get("goodsName", ""),
                        "정가":     item.get("normalPrice", 0),
                        "판매가":   item.get("price", 0),
                        "할인율":   item.get("saleRate", 0),
                        "리뷰수":   item.get("reviewCount", 0),
                        "리뷰점수": item.get("reviewScore", 0),
                        "판매상태": "SOLD_OUT" if is_sold_out else "SALE",
                    })

                pagination = data.get("pagination", {})
                total_pages = pagination.get("totalPages", 1)
                print(f"상품 수집 중... {page}/{total_pages} 페이지 (누적 {len(all_data)}개)")

                next_page_url = pagination.get("nextPageUrl")

                if not pagination.get("hasNext", False) or page >= total_pages:
                    break
                page += 1
                time.sleep(2)

            except Exception as e:
                print(f"상품 수집 오류 (page {page}): {e}")
                time.sleep(2)
                break

        df = pd.DataFrame(all_data)
        df = df.sort_values("리뷰수", ascending=False).reset_index(drop=True)
        df["조회수"] = 0
        df["누적판매수"] = 0
        print(f"\n상품 총 {len(df)}개 수집 완료 (리뷰 많은 순)")
        print(df[["카테고리", "브랜드", "상품명", "리뷰수"]].to_string(index=False))

        collected_review_ids = set()

        if os.path.exists(PRODUCTS_CSV):
            df_existing = pd.read_csv(PRODUCTS_CSV)
            df_existing["goodsNo"] = df_existing["goodsNo"].astype(str)
            df_existing.set_index("goodsNo", inplace=True)
            print(f"\n기존 수집된 상품 {len(df_existing)}개 확인 -> 기존 정보 최신화 및 신규 리뷰 수집 예정")
        else:
            df_existing = pd.DataFrame()
            print("\n기존 수집 상품 파일 없음 -> 처음부터 수집 시작")

        if os.path.exists(REVIEWS_CSV):
            df_existing_reviews = pd.read_csv(REVIEWS_CSV)
            collected_review_ids = set(df_existing_reviews["리뷰번호"].astype(str).tolist())
            print(f"기존 수집된 개별 리뷰 {len(collected_review_ids)}개 확인 -> 중복 리뷰 스킵 예정")

        total_items = len(df)

        for idx, row in df.iterrows():
            goods_no     = row["goodsNo"]
            target_count = int(row["리뷰수"])
            current_item = idx + 1

            is_new_product = df_existing.empty or goods_no not in df_existing.index
            print(f"\n[{current_item}/{total_items}] 처리 중: {str(row['상품명'])[:25]} (총 리뷰 수: {target_count}개)")

            try:
                sales, views = get_goods_stats(goods_no)
            except Exception as e:
                print(f"  통계 수집 실패: {e}")
                save_error(goods_no, "stats", str(e))
                if not is_new_product:
                    sales = df_existing.at[goods_no, "누적판매수"]
                    views = df_existing.at[goods_no, "조회수"]
                else:
                    sales, views = 0, 0
            time.sleep(2)

            updated_info = {
                "플랫폼":    row["플랫폼"],
                "카테고리":  row["카테고리"],
                "브랜드":    row["브랜드"],
                "상품명":    row["상품명"],
                "정가":      row["정가"],
                "판매가":    row["판매가"],
                "할인율":    row["할인율"],
                "조회수":    views,
                "누적판매수": sales,
                "리뷰수":    row["리뷰수"],
                "리뷰점수":  row["리뷰점수"],
                "판매상태":  row["판매상태"],
            }

            for key, value in updated_info.items():
                df_existing.at[goods_no, key] = value

            if is_new_product:
                print(f"  -> 신규 상품 저장 완료 (누적판매: {sales} | 조회수: {views})")
            else:
                print(f"  -> 기존 상품 정보 최신화 완료 (누적판매: {sales} | 조회수: {views})")

            if not goods_no or target_count == 0:
                print(f"  [스킵] 리뷰 없음")
            else:
                print(f"  새로운 리뷰 수집 확인 중...")
                try:
                    reviews = get_reviews(goods_no, target_count=target_count, collected_ids=collected_review_ids)
                    if len(reviews) > 0:
                        save_reviews(reviews)
                        collected_review_ids.update([str(r["리뷰번호"]) for r in reviews])
                        print(f"  -> 새로운 리뷰 {len(reviews)}개 수집 및 추가 완료!")
                    else:
                        print(f"  -> 추가된 새로운 리뷰가 없습니다.")
                except Exception as e:
                    print(f"  리뷰 수집 실패: {e}")
                    save_error(goods_no, "review", str(e))

            time.sleep(2)

        df_existing.reset_index(inplace=True)
        if 'index' in df_existing.columns:
            df_existing.rename(columns={'index': 'goodsNo'}, inplace=True)
        df_existing.to_csv(PRODUCTS_CSV, index=False, encoding="utf-8-sig")
        print(f"\n[{BRAND} / {CATEGORY_NAME}] 모든 상품 정보 최신화 및 CSV 저장 완료!")

print("\n수집 완료!")


▶ 브랜드 [jemut] 수집 시작

▶ 카테고리 [001] 상의 수집 시작
상품 수집 중... 1/37 페이지 (누적 30개)
상품 수집 중... 2/37 페이지 (누적 60개)
상품 수집 중... 3/37 페이지 (누적 90개)
상품 수집 중... 4/37 페이지 (누적 120개)
상품 수집 중... 5/37 페이지 (누적 150개)
상품 수집 중... 6/37 페이지 (누적 180개)
상품 수집 중... 7/37 페이지 (누적 210개)
상품 수집 중... 8/37 페이지 (누적 240개)
상품 수집 중... 9/37 페이지 (누적 270개)
상품 수집 중... 10/37 페이지 (누적 300개)
상품 수집 중... 11/37 페이지 (누적 330개)
상품 수집 중... 12/37 페이지 (누적 360개)
상품 수집 중... 13/37 페이지 (누적 390개)
상품 수집 중... 14/37 페이지 (누적 420개)
상품 수집 중... 15/37 페이지 (누적 450개)
상품 수집 중... 16/37 페이지 (누적 480개)
상품 수집 중... 17/37 페이지 (누적 510개)
상품 수집 중... 18/37 페이지 (누적 540개)
상품 수집 중... 19/37 페이지 (누적 570개)
상품 수집 중... 20/37 페이지 (누적 600개)
상품 수집 중... 21/37 페이지 (누적 630개)
상품 수집 중... 22/37 페이지 (누적 660개)
상품 수집 중... 23/37 페이지 (누적 690개)
상품 수집 중... 24/37 페이지 (누적 720개)
상품 수집 중... 25/37 페이지 (누적 750개)
상품 수집 중... 26/37 페이지 (누적 780개)
상품 수집 중... 27/37 페이지 (누적 810개)
상품 수집 중... 28/37 페이지 (누적 840개)
상품 수집 중... 29/37 페이지 (누적 870개)
상품 수집 중... 30/37 페이지 (누적 900개)
상품 수집 중... 31/37 페이지 (누적 930개)
상품 수집 중

In [4]:
import pandas as pd

file_names = [
    "data/filluminate_products_001.csv",
    "data/filluminate_products_002.csv",
    "data/filluminate_products_003.csv",
    "data/travel_products_001.csv",
    "data/travel_products_002.csv",
    "data/travel_products_003.csv",
    "data/jemut_products_001.csv",
    "data/jemut_products_002.csv",
    "data/jemut_products_003.csv",
    "data/filluminate_reviews_001.csv",
    "data/filluminate_reviews_002.csv",
    "data/filluminate_reviews_003.csv",
    "data/travel_reviews_001.csv",
    "data/travel_reviews_002.csv",
    "data/travel_reviews_003.csv",
    "data/jemut_reviews_001.csv",
    "data/jemut_reviews_002.csv",
    "data/jemut_reviews_003.csv",
]

for file in file_names:
    try:
        # 파일 읽기
        df = pd.read_csv(file)
        
        # 이전 행 개수 확인 (진행 상황 확인용)
        before_count = len(df)
        
        # 완전 중복 제거 (모든 컬럼의 값이 동일한 행 중 하나만 남김)
        df.drop_duplicates(inplace=True)
        
        # 삭제 후 행 개수 확인
        after_count = len(df)
        
        # 중복이 제거된 데이터프레임을 원본 파일에 덮어쓰기
        df.to_csv(file, index=False)
        print(f"{file} 완료 (중복 {before_count - after_count}개 제거됨)")
        
    except FileNotFoundError:
        print(f"{file} 파일을 찾을 수 없습니다. 경로를 확인해 주세요.")
    except Exception as e:
        print(f"{file} 처리 중 에러 발생: {e}")

print("모든 파일의 중복 제거 및 업데이트 완료")

data/filluminate_products_001.csv 완료 (중복 0개 제거됨)
data/filluminate_products_002.csv 완료 (중복 0개 제거됨)
data/filluminate_products_003.csv 완료 (중복 0개 제거됨)
data/travel_products_001.csv 완료 (중복 0개 제거됨)
data/travel_products_002.csv 완료 (중복 0개 제거됨)
data/travel_products_003.csv 완료 (중복 0개 제거됨)
data/jemut_products_001.csv 완료 (중복 0개 제거됨)
data/jemut_products_002.csv 완료 (중복 0개 제거됨)
data/jemut_products_003.csv 완료 (중복 0개 제거됨)
data/filluminate_reviews_001.csv 완료 (중복 0개 제거됨)
data/filluminate_reviews_002.csv 완료 (중복 0개 제거됨)
data/filluminate_reviews_003.csv 완료 (중복 0개 제거됨)
data/travel_reviews_001.csv 완료 (중복 0개 제거됨)
data/travel_reviews_002.csv 완료 (중복 0개 제거됨)
data/travel_reviews_003.csv 완료 (중복 0개 제거됨)
data/jemut_reviews_001.csv 완료 (중복 0개 제거됨)
data/jemut_reviews_002.csv 완료 (중복 0개 제거됨)
data/jemut_reviews_003.csv 완료 (중복 0개 제거됨)
모든 파일의 중복 제거 및 업데이트 완료
